In [ ]:
import pandas as pd, numpy as np
import vivarium_inputs
import gbd_mapping
import pathlib

In [ ]:
location = "India"
vehicle = "rice"

In [ ]:
location = location.title()

In [ ]:
pop = vivarium_inputs.get_population_structure(location).value
pop[pop > 0]

In [ ]:
asfr = vivarium_inputs.get_measure(
    gbd_mapping.covariates.age_specific_fertility_rate, "estimate", location
).value
asfr[asfr > 0]

In [ ]:
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)
asfr

In [ ]:
sbr = vivarium_inputs.get_measure(
    gbd_mapping.covariates.stillbirth_to_live_birth_ratio, "estimate", location
).value
sbr[sbr > 0]

In [ ]:
sbr = sbr.iloc[0]
sbr

In [ ]:
maternal_abortion_miscarriage_incidence = vivarium_inputs.get_measure(
    gbd_mapping.causes.maternal_abortion_and_miscarriage,
    "incidence_rate",
    location,
).mean(axis=1)
maternal_abortion_miscarriage_incidence[maternal_abortion_miscarriage_incidence > 0]

In [ ]:
ectopic_pregnancy_incidence = vivarium_inputs.get_measure(
    gbd_mapping.causes.ectopic_pregnancy,
    "incidence_rate",
    location,
).mean(axis=1)
ectopic_pregnancy_incidence[ectopic_pregnancy_incidence > 0]

In [ ]:
pregnancy_incidence = (
    asfr
    + (asfr * sbr)
    + maternal_abortion_miscarriage_incidence
    + ectopic_pregnancy_incidence
)
pregnancy_incidence[pregnancy_incidence > 0]

In [ ]:
pregnancies = pop * pregnancy_incidence
pregnancies[pregnancies > 0]

In [ ]:
total_pregnancies = pregnancies.sum()
total_pregnancies

In [ ]:
sim_population = (
    pd.read_parquet(
        f"../../0200_pregnancy_sim/sim_results/{vehicle}/{location.lower()}/pregnancy_outcome_count.parquet"
    )
    .groupby(["input_draw", "scenario"])
    .value.sum()
    .mean()
)
sim_population

In [ ]:
scalar = total_pregnancies / sim_population
scalar

In [ ]:
for result in [
    "ylds",
    "ylls",
    "pregnancy_outcome_count",
    "person_time_anemia",
    "transition_count_maternal_disorders",
]:
    df = pd.read_parquet(
        f"../../0200_pregnancy_sim/sim_results/{vehicle}/{location.lower()}/{result}.parquet"
    )
    df.value *= scalar
    path = pathlib.Path(
        f"../results/rescaled_pregnancy_results/{vehicle}/{location.lower()}/{result}.parquet"
    )
    path.parent.mkdir(exist_ok=True, parents=True)
    df.to_parquet(path)